# Swarm Intelligence: Phase 1 - Environment Data Synthesis & Quality Control

## Overview
This notebook executes the complete Exploratory Data Analysis (EDA), cleaning, feature selection, and temporal synthesis pipeline for Phase 1 of the **Swarm Intelligence Framework**.

### Objective
To build a mathematically locked, continuous 8,784-hour state-space matrix for the year 2020 by fusing localized meteorological data (**NREL NSRDB**) with regional electrical human demand (**ERCOT Coast Zone**).

### Data Inputs Required in Kaggle Input Directory:
1. `nsrdb-houston-2020.csv` — Hourly meteorological and solar data for Houston, TX.
2. `ercot-2020.xlsx` — ERCOT Hourly Native Load report for Texas regional zones.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Define file paths (Adjust filenames if your Kaggle input path differs)
NREL_PATH = '/kaggle/input/datasets/vsnihal/houston2020/nsrdb-houston-2020.csv'  # or local path 'nsrdb-houston-2020.csv'
ERCOT_PATH = '/kaggle/input/datasets/vsnihal/houston2020/ercot-2020.xlsx'        # or local path 'ercot-2020.xlsx'

# If running locally, check current working directory
if not os.path.exists(NREL_PATH):
    NREL_PATH = 'nsrdb-houston-2020.csv'
if not os.path.exists(ERCOT_PATH):
    ERCOT_PATH = 'ercot-2020.xlsx'

print("Setup completed successfully.")

## 1. Exploratory Data Analysis (EDA) on Raw Datasets

Before transforming the data, we must inspect the raw structural signatures of both datasets to uncover:
1. Header offsets and metadata lines.
2. Column names, types, and null counts.
3. Temporal coverage and row counts (2020 is a leap year = 366 days * 24 hours = **8,784 expected rows**).

In [ ]:
# 1. Inspect Raw NREL Structure
print("=== NREL RAW METADATA & HEADERS ===")
with open(NREL_PATH, 'r') as f:
    for i in range(4):
        print(f"Line {i}: {f.readline().strip()}")

# Load NREL skipping the 2 metadata header rows
df_nrel_raw = pd.read_csv(NREL_PATH, skiprows=2)
print(f"\nRaw NREL Shape: {df_nrel_raw.shape}")
print(f"NREL Raw Columns: {list(df_nrel_raw.columns)}")

# 2. Inspect Raw ERCOT Structure
print("\n=== ERCOT RAW STRUCTURE ===")
xl_ercot = pd.ExcelFile(ERCOT_PATH)
print(f"Available Sheets in Excel File: {xl_ercot.sheet_names}")

df_ercot_raw = pd.read_excel(ERCOT_PATH, sheet_name='Native Load Report')
print(f"Raw ERCOT Shape: {df_ercot_raw.shape}")
print(f"ERCOT Raw Columns: {list(df_ercot_raw.columns)}")
print("\nFirst 3 rows of Raw ERCOT:")
print(df_ercot_raw.head(3))

## 2. NREL Cleaning & Feature Engineering

### Feature Selection Rationale
Raw NSRDB files contain numerous redundant parameters (e.g., aerosol optical depth, clearsky theoretical baselines, surface albedo). To avoid feeding noise into our Reinforcement Learning state-space:
* **Retained Attributes:**
  * `GHI` (Global Horizontal Irradiance): Total solar irradiance on a horizontal surface.
  * `DNI` (Direct Normal Irradiance): Direct beam solar radiation.
  * `DHI` (Diffuse Horizontal Irradiance): Scattered sky radiation.
  * `Temperature`: Ambient surface temperature (drives HVAC power demand & impacts battery chemical efficiency).
  * `Wind Speed`: Secondary renewable generation indicator.
* **Dropped Columns:** All metadata, clearsky baselines, and cloud classification flags.

### Leap Year Anomaly Resolution
2020 was a leap year (366 days = 8,784 hours). The raw NREL file contains only 8,760 rows because NSRDB standard downloads omit February 29th.
* **Resolution:** We explicitly identify the missing February 29th records and impute them by borrowing temporal weather profiles from February 28th, maintaining continuity across the full annual cycle.

In [ ]:
# Create clean working copy
df_nrel = pd.read_csv(NREL_PATH, skiprows=2)

# Build unified datetime index
df_nrel['timestamp'] = pd.to_datetime(
    df_nrel[['Year', 'Month', 'Day', 'Hour', 'Minute']]
)
df_nrel['timestamp'] = df_nrel['timestamp'].dt.floor('h').dt.tz_localize(None)

# Select only necessary state-space features
selected_nrel_cols = ['timestamp', 'GHI', 'DNI', 'DHI', 'Temperature', 'Wind Speed']
df_nrel = df_nrel[selected_nrel_cols].copy()

print(f"NREL shape before Leap Year fix: {df_nrel.shape}")

# Check for Feb 29 presence
feb29_rows = df_nrel[df_nrel['timestamp'].dt.date == pd.to_datetime('2020-02-29').date()]

if len(feb29_rows) == 0:
    print("WARNING: Feb 29 missing in NREL data. Synthesizing Feb 29 from Feb 28 baseline...")
    # Extract Feb 28 data and shift forward by 1 day
    feb28_data = df_nrel[df_nrel['timestamp'].dt.date == pd.to_datetime('2020-02-28').date()].copy()
    feb28_data['timestamp'] = feb28_data['timestamp'] + pd.Timedelta(days=1)
    
    # Append and re-sort
    df_nrel = pd.concat([df_nrel, feb28_data]).sort_values('timestamp').reset_index(drop=True)

print(f"NREL shape after Leap Year fix: {df_nrel.shape} (Target: 8,784 rows)")
print("\nCleaned NREL Head:")
print(df_nrel.head(3))

## 3. ERCOT Cleaning & Temporal Alignment

### Spatial Filtering Rationale
Texas ERCOT covers 8 distinct load zones. To align human load with our NREL weather data centered on Houston, we extract **only the `COAST` load zone** and drop all other regions (`NORTH`, `SOUTH`, `WEST`, etc.).

### String Scrubbing & Temporal Adjustments
1. **Daylight Saving Scrub:** Remove `' DST'` tags from timestamp strings.
2. **Midnight Mapping:** ERCOT records midnight using the non-standard `24:00` format (e.g., `01/01/2020 24:00`). We convert `24:00` to `00:00` of the following day.
3. **1-Hour Shift Alignment:** 
   * ERCOT reports demand as **"Hour Ending"** (e.g., `01:00` represents energy consumed from `00:00` to `01:00`).
   * NREL reports weather at the **"Interval Start"** (`00:00`).
   * To align human demand with the exact weather conditions that generated it, we subtract 1 hour from ERCOT timestamps.

In [ ]:
df_ercot = pd.read_excel(ERCOT_PATH, sheet_name='Native Load Report')

# Retain only HourEnding and COAST zone demand
df_ercot = df_ercot[['HourEnding', 'COAST']].copy()
df_ercot.rename(columns={'COAST': 'demand_mw'}, inplace=True)

# DST ANOMALY BYPASS: 
# ERCOT provides exactly 8,784 sequential rows but uses messy DST string labels.
# We drop their labels and forcefully apply a continuous, Standard Time 8,784-hour index.
df_ercot['timestamp'] = pd.date_range(
    start='2020-01-01 00:00:00', 
    periods=8784, 
    freq='h'
)

# Drop the messy raw string column
df_ercot.drop(columns=['HourEnding'], inplace=True)

# Reorder columns
df_ercot = df_ercot[['timestamp', 'demand_mw']]

print(f"Cleaned ERCOT Shape: {df_ercot.shape} (Target: 8,784 rows)")
print(f"Duplicates in ERCOT: {df_ercot['timestamp'].duplicated().sum()}")
print("\nCleaned ERCOT Head:")
print(df_ercot.head(3))

## 4. Master Environment Synthesis

We perform an **Inner Join** between the processed NREL dataframe and the ERCOT dataframe on the exact hourly `timestamp`. 

This produces a single unified dataframe where every timestamp connects meteorological state variables (solar/temp/wind) directly with human electricity demand.

In [ ]:
# Inner Merge on exact timestamp
df_master = pd.merge(df_nrel, df_ercot, on='timestamp', how='inner')
df_master = df_master.sort_values('timestamp').reset_index(drop=True)

print(f"Master Environment Shape: {df_master.shape}")

# Re-order columns logically
master_cols = ['timestamp', 'GHI', 'DNI', 'DHI', 'Temperature', 'Wind Speed', 'demand_mw']
df_master = df_master[master_cols]

# Export to CSV
OUTPUT_FILE = 'phase_01_master_environment_full.csv'
df_master.to_csv(OUTPUT_FILE, index=False)

print(f"Successfully generated master state-space environment: '{OUTPUT_FILE}'")
print("\nMaster Dataset Preview:")
display(df_master.head(5))

## 5. Automated Data Quality Verification & Unit Tests

To guarantee the state-space matrix is robust enough to feed into MARL simulation loops without causing execution crashes, we run 5 automated verification checks:
1. **Row Count Validation:** Must equal exactly 8,784 hours.
2. **Temporal Continuity Test:** Ensures zero missing hourly steps.
3. **Null/NaN Check:** Ensures no incomplete numerical records.
4. **Duplicate Index Test:** Guarantees unique hourly records.
5. **Distribution Analysis:** Visualizes seasonal variance across 2020.

In [ ]:
print("=== RUNNING QA UNIT TESTS ===")

# Test 1: Total Row Count Check
expected_rows = 8784
assert len(df_master) == expected_rows, f"FAIL: Row count is {len(df_master)}, expected {expected_rows}"
print(f" [PASS] Test 1: Row count hits exact target of {expected_rows} hours.")

# Test 2: Null / Missing Values Check
null_count = df_master.isnull().sum().sum()
assert null_count == 0, f"FAIL: Found {null_count} missing values."
print(" [PASS] Test 2: Zero null or missing values detected.")

# Test 3: Duplicate Timestamp Check
duplicate_count = df_master['timestamp'].duplicated().sum()
assert duplicate_count == 0, f"FAIL: Found {duplicate_count} duplicate timestamps."
print(" [PASS] Test 3: Zero duplicate timestamps detected.")

# Test 4: Time Continuity Check (Fixed 1-hour step difference)
time_diffs = df_master['timestamp'].diff()[1:]
expected_step = pd.Timedelta(hours=1)
missing_steps = (time_diffs != expected_step).sum()
assert missing_steps == 0, f"FAIL: Found {missing_steps} gaps in the hourly timeline."
print(" [PASS] Test 4: 100% continuous hourly temporal sequence verified.")

# Test 5: Summary Statistics Overview
print("\n=== MASTER STATE-SPACE STATISTICAL SUMMARY ===")
display(df_master.describe().T[['mean', 'std', 'min', '50%', 'max']])

## 6. Visual Exploratory Data Analysis

We plot the yearly seasonal trends for Demand vs Solar Irradiance (`GHI`) and Temperature to verify physical relationships before building the swarm environment.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: ERCOT COAST Demand
axes[0].plot(df_master['timestamp'], df_master['demand_mw'], color='crimson', alpha=0.85, linewidth=0.8)
axes[0].set_title('2020 ERCOT Coast Zone Electricity Demand (MW)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Demand (MW)')

# Plot 2: Solar Global Horizontal Irradiance (GHI)
axes[1].plot(df_master['timestamp'], df_master['GHI'], color='gold', alpha=0.85, linewidth=0.8)
axes[1].set_title('2020 NREL Solar GHI (W/m²)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('GHI (W/m²)')

# Plot 3: Temperature Profile
axes[2].plot(df_master['timestamp'], df_master['Temperature'], color='teal', alpha=0.85, linewidth=0.8)
axes[2].set_title('2020 Ambient Temperature (°C)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Temp (°C)')
axes[2].set_xlabel('Timestamp')

plt.tight_layout()
plt.show()

## 7. Multi-Agent Gymnasium Environment (`GridEnv`)

This section translates our 8,784-hour state-space matrix into a dynamic, step-by-step reinforcement learning environment. 

### Core Mechanics
* **Agents:** We simulate a decentralized swarm of `num_agents` (e.g., 10 independent microgrids).
* **Observation ($s_t$):** Each agent sees the global weather and grid demand, plus its own localized battery State of Charge (SoC).
* **Action ($a_t$):** Continuous control `[-1.0, 1.0]`. `-1.0` is maximum discharge (selling to the grid), `1.0` is maximum charge (pulling from the grid), and `0` is idle.
* **Reward ($R_t$):** Agents receive a shared, cooperative reward based on how effectively their combined actions flatten the grid's demand curve.

In [ ]:
# 1. Force install the modern RL library
!pip install gymnasium

# 2. Re-import critical dependencies to ensure the cell is self-contained
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class GridEnv(gym.Env):
    """
    Custom Environment for Swarm Intelligence Grid Balancing.
    Simulates independent battery agents interacting with the ERCOT/NREL matrix.
    """
    metadata = {'render_modes': ['human']}

    def __init__(self, data_path='phase_01_master_environment_full.csv', num_agents=10):
        super(GridEnv, self).__init__()
        
        # Load Phase 1 Master Environment Matrix
        self.df = pd.read_csv(data_path)
        self.max_steps = len(self.df) - 1
        self.current_step = 0
        self.num_agents = num_agents
        
        # Isolate numerical features: GHI, DNI, DHI, Temperature, Wind Speed, demand_mw
        self.feature_cols = ['GHI', 'DNI', 'DHI', 'Temperature', 'Wind Speed', 'demand_mw']
        self.num_global_features = len(self.feature_cols)
        
        # Observation Space: Global Features + 1 Local Feature (State of Charge)
        self.observation_space = spaces.Dict({
            f'agent_{i}': spaces.Box(low=-np.inf, high=np.inf, shape=(self.num_global_features + 1,), dtype=np.float32)
            for i in range(self.num_agents)
        })
        
        # Action Space: Continuous control [-1.0 (Discharge) to 1.0 (Charge)]
        self.action_space = spaces.Dict({
            f'agent_{i}': spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
            for i in range(self.num_agents)
        })
        
        # Swarm Physical Parameters
        self.agent_soc = np.zeros(self.num_agents)  # State of Charge [0.0 to 1.0]
        self.charge_rate = 0.1                      # Max 10% charge/discharge per hour
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        # Initialize all batteries at 50% capacity
        self.agent_soc = np.full(self.num_agents, 0.5, dtype=np.float32) 
        
        return self._get_obs(), {}

    def _get_obs(self):
        """Constructs the observation vector for each agent at step t."""
        row = self.df.iloc[self.current_step]
        global_state = np.array([row[col] for col in self.feature_cols], dtype=np.float32)
        
        obs = {}
        for i in range(self.num_agents):
            # Agent sees the global state + its personal State of Charge
            obs[f'agent_{i}'] = np.append(global_state, self.agent_soc[i]).astype(np.float32)
        return obs

    def step(self, action_dict):
        """Executes the swarm's decisions and calculates grid physics."""
        
        # 1. Baseline Grid State
        current_demand = self.df.iloc[self.current_step]['demand_mw']
        
        # 2. Agent Dynamics
        total_grid_impact = 0.0
        for i in range(self.num_agents):
            action = action_dict[f'agent_{i}'][0]
            
            # Calculate physical power movement
            power_delta = action * self.charge_rate
            
            # Apply to battery (clipping ensures we don't overcharge/over-drain)
            new_soc = np.clip(self.agent_soc[i] + power_delta, 0.0, 1.0)
            actual_power_moved = new_soc - self.agent_soc[i]
            
            self.agent_soc[i] = new_soc
            total_grid_impact += actual_power_moved * 100  # Scaling factor: Assuming 100MW microgrid capacities
            
        # 3. Resolve New Grid Load
        # Charging adds to demand; discharging subtracts from demand.
        modified_demand = current_demand + total_grid_impact
        
        # 4. Cooperative Reward Engineering
        # Goal: Flatten the curve. We penalize the system based on how far it deviates from an ideal baseline.
        ideal_baseline = 12000.0  # MW target threshold
        reward_value = -abs(modified_demand - ideal_baseline)
        
        rewards = {f'agent_{i}': reward_value for i in range(self.num_agents)}
        
        # 5. Temporal Advancement
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False
        
        dones = {f'agent_{i}': terminated for i in range(self.num_agents)}
        dones['__all__'] = terminated
        
        infos = {f'agent_{i}': {'modified_demand': modified_demand} for i in range(self.num_agents)}
        
        return self._get_obs(), rewards, terminated, truncated, infos

## 8. Gymnasium Environment Sanity Check & 24-Hour Simulation Test

To verify that the `GridEnv` class operates correctly before hooking up actual RL agents (e.g., PPO or MAPPO), we run a 24-hour diagnostic simulation.

### Diagnostic Objectives
1. Verify `env.reset()` returns the correct state dictionary for all 10 agents.
2. Simulate random agent actions (`charge` vs `discharge`) across 24 hourly steps.
3. Validate that battery States of Charge ($\text{SoC}$) stay strictly bounded in `[0.0, 1.0]`.
4. Compare baseline grid demand vs swarm-modified grid demand.

In [ ]:
# 1. Instantiate the environment
env = GridEnv(data_path='phase_01_master_environment_full.csv', num_agents=10)

# 2. Reset the environment
obs, info = env.reset()

print("=== ENVIRONMENT INITIALIZATION ===")
print(f"Active Agents: {env.num_agents}")
print(f"Single Agent Observation Shape: {obs['agent_0'].shape}")
print(f"Agent 0 Initial State (GHI, DNI, DHI, Temp, Wind, Demand, SoC):\n{obs['agent_0']}\n")

# 3. Run a 24-hour simulation loop using random actions
history_baseline = []
history_modified = []
history_rewards = []

for step in range(24):
    # Sample random actions [-1.0, 1.0] for all agents
    action_dict = {f'agent_{i}': env.action_space[f'agent_{i}'].sample() for i in range(env.num_agents)}
    
    # Store baseline demand for plotting
    baseline_d = env.df.iloc[env.current_step]['demand_mw']
    history_baseline.append(baseline_d)
    
    # Execute environment step
    obs, rewards, terminated, truncated, infos = env.step(action_dict)
    
    # Track modified demand and rewards
    modified_d = infos['agent_0']['modified_demand']
    history_modified.append(modified_d)
    history_rewards.append(rewards['agent_0'])

print("=== 24-HOUR DIAGNOSTIC TEST RESULT ===")
print(f"Hour 0 Baseline Demand: {history_baseline[0]:.2f} MW | Modified: {history_modified[0]:.2f} MW")
print(f"24h Peak Demand: Baseline = {max(history_baseline):.2f} MW | Modified = {max(history_modified):.2f} MW")
print(f"Average Cooperative Reward per Step: {np.mean(history_rewards):.2f}")

In [ ]:
# Plot Baseline vs Swarm Modified Demand for the 24-hour test window
plt.figure(figsize=(12, 5))
plt.plot(range(24), history_baseline, label='Original Grid Demand (MW)', color='crimson', linestyle='--', linewidth=2)
plt.plot(range(24), history_modified, label='Swarm Modified Demand (MW)', color='navy', linewidth=2)
plt.axhline(12000, color='gray', linestyle=':', label='Target Baseline (12,000 MW)')

plt.title('24-Hour Diagnostic Run: Random Swarm Action Impact', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=10)
plt.ylabel('Demand (MW)', fontsize=10)
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

# Phase 3: Swarm Intelligence Training (Independent PPO)

## Overview
Now that the environment physics are locked, we deploy **Independent Proximal Policy Optimization (IPPO)** across the swarm. 

### Why IPPO?
1. **Decentralized Execution:** Each agent acts autonomously using its local observation vector without requiring a central controller during runtime.
2. **Continuous Action Control:** PPO smoothly outputs continuous battery charge/discharge rates in `[-1.0, 1.0]`.
3. **Training Stability:** Clipped surrogate objective functions prevent destructive policy updates when adjusting battery schedules across 8,784 seasonal hours.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing Full Annual Production Run on: {device}")

# =================================================================
# 1. UPGRADED ENVIRONMENT PHYSICS & ACTOR-CRITIC
# =================================================================
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim=1):
        super(ActorCritic, self).__init__()
        self.fc_shared = nn.Sequential(
            nn.Linear(state_dim, 128), nn.Tanh(),
            nn.Linear(128, 64), nn.Tanh()
        )
        self.actor_mean = nn.Linear(64, action_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(1, action_dim))
        self.critic = nn.Linear(64, 1)

    def forward(self, state):
        x = self.fc_shared(state)
        mean = torch.tanh(self.actor_mean(x))
        std = torch.exp(self.actor_log_std).expand_as(mean)
        return Normal(mean, std), self.critic(x)

def upgraded_step(self, action_dict):
    current_demand = self.df.iloc[self.current_step]['demand_mw']
    total_grid_impact = 0.0
    
    for i in range(self.num_agents):
        action = action_dict[f'agent_{i}']
        power_delta = action * self.charge_rate
        new_soc = np.clip(self.agent_soc[i] + power_delta, 0.0, 1.0)
        actual_power_moved = new_soc - self.agent_soc[i]
        self.agent_soc[i] = new_soc
        
        # UPGRADE: 500 MW capacity per agent
        total_grid_impact += actual_power_moved * 500  

    modified_demand = current_demand + total_grid_impact
    
    # UPGRADE: Centered baseline to force cyclic charging/discharging
    ideal_baseline = 10500.0
    
    # NORMALIZATION: Scaled rewards
    reward_value = -abs(modified_demand - ideal_baseline) / 1000.0
    rewards = {f'agent_{i}': float(reward_value) for i in range(self.num_agents)}
    
    self.current_step += 1
    terminated = self.current_step >= self.max_steps
    infos = {f'agent_{i}': {'modified_demand': float(modified_demand)} for i in range(self.num_agents)}
    
    return self._get_obs(), rewards, terminated, False, infos

GridEnv.step = upgraded_step
env = GridEnv()

# =================================================================
# 2. FULL ANNUAL TRAINING LOOP (WITH CHECKPOINTING)
# =================================================================
NUM_EPISODES = 50         # 50 full years of experience
EPISODE_STEPS = env.max_steps  # 8,784 steps
GAMMA = 0.99
policy = ActorCritic(state_dim=7, action_dim=1).to(device)

# Reverting to the stable learning rate for long-term optimization
optimizer = optim.Adam(policy.parameters(), lr=3e-4)

obs_scaler = torch.tensor([1044.0, 1019.0, 554.0, 40.0, 10.0, 21000.0, 1.0], dtype=torch.float32).to(device)

os.makedirs("checkpoints", exist_ok=True)
print("\n=== INITIATING SWARM INTELLIGENCE: FULL ANNUAL TRAINING ===")

for episode in range(1, NUM_EPISODES + 1):
    obs, _ = env.reset()
    total_episode_reward = 0.0
    states, actions, log_probs, rewards_list, values = [], [], [], [], []
    
    for step in range(EPISODE_STEPS):
        action_dict = {}
        s_states, s_actions, s_log_probs, s_vals = [], [], [], []
        
        for i in range(env.num_agents):
            agent_obs = (torch.tensor(obs[f'agent_{i}'], dtype=torch.float32).unsqueeze(0).to(device)) / obs_scaler
            dist, val = policy(agent_obs)
            action = torch.clamp(dist.sample(), -1.0, 1.0)
            
            action_dict[f'agent_{i}'] = action.item() 
            
            s_states.append(agent_obs)
            s_actions.append(action)
            s_log_probs.append(dist.log_prob(action))
            s_vals.append(val)
            
        obs, rewards, terminated, _, _ = env.step(action_dict)
        step_reward = rewards['agent_0']
        total_episode_reward += step_reward
        
        states.append(torch.stack(s_states))
        actions.append(torch.stack(s_actions))
        log_probs.append(torch.stack(s_log_probs))
        values.append(torch.stack(s_vals))
        rewards_list.append(step_reward)
        
        if terminated: break

    # Generalized Advantage Estimation
    returns, R = [], 0
    for r in reversed(rewards_list):
        R = r + GAMMA * R
        returns.insert(0, R)
    returns = torch.tensor(returns, dtype=torch.float32).to(device)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    policy_loss = 0.0
    for t in range(len(rewards_list)):
        for i in range(env.num_agents):
            adv = returns[t] - values[t][i].squeeze()
            policy_loss += -log_probs[t][i].squeeze() * adv.detach() + 0.5 * adv.pow(2)
            
    optimizer.zero_grad()
    policy_loss.backward()
    optimizer.step()
    
    avg_reward = total_episode_reward / EPISODE_STEPS
    print(f"Year {episode:02d}/{NUM_EPISODES} | Avg Reward: {avg_reward:.3f} | Loss: {policy_loss.item():.2f}")
    
    # Save checkpoint every 10 episodes
    if episode % 10 == 0:
        torch.save(policy.state_dict(), f"checkpoints/swarm_policy_ep{episode}.pth")
        print(f"--> Checkpoint saved: checkpoints/swarm_policy_ep{episode}.pth")

print("\n=== SWARM TRAINING COMPLETE ===")

# =================================================================
# 3. 7-DAY VISUAL EVALUATION PLOT
# =================================================================
# Plotting a full year is a solid block of ink, so we plot a 1-week slice
EVAL_STEPS = 168 # 7 Days * 24 Hours
obs, _ = env.reset()
eval_baseline, eval_modified = [], []

for step in range(EVAL_STEPS): 
    action_dict = {}
    for i in range(env.num_agents):
        agent_obs = (torch.tensor(obs[f'agent_{i}'], dtype=torch.float32).unsqueeze(0).to(device)) / obs_scaler
        with torch.no_grad():
            action_dict[f'agent_{i}'] = policy(agent_obs)[0].mean.item()
            
    eval_baseline.append(env.df.iloc[env.current_step]['demand_mw'])
    obs, _, _, _, infos = env.step(action_dict)
    eval_modified.append(infos['agent_0']['modified_demand'])

plt.figure(figsize=(16, 6))
plt.plot(range(EVAL_STEPS), eval_baseline, label='Unmitigated Grid Demand', color='crimson', linestyle='--', linewidth=2)
plt.plot(range(EVAL_STEPS), eval_modified, label='Swarm Managed Demand', color='forestgreen', linewidth=2)
plt.axhline(10500, color='gray', linestyle=':', label='Target Baseline (10.5 GW)')
plt.title('Production Validation: 7-Day Swarm Peak Shaving', fontweight='bold')
plt.xlabel('Hour of the Week', fontsize=10)
plt.ylabel('Demand (MW)', fontsize=10)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()